# Análise Exploratória de Dados (EDA)

Este notebook realiza análises descritivas sobre os dados estruturados de influenciadores.
Ele reflete a nova arquitetura de camadas do processamento:

- **`base_2017_2019`**: Para análises descritivas gerais. Não exclui zeros nos comentários para evitar viés.
- **`core_2017_2019`**: Para análises de robustez e estabilidade, com os zeros excluídos.

## 1. Configuração

In [33]:
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, pearsonr

# Caminhos
PARQUET_BASE_2017_2019 = "posts_base_2017_2019.parquet"
PARQUET_CORE_2017_2019 = "posts_core_2017_2019.parquet"
PARQUET_CV_CORE = "cv_core_2017_2019.parquet"
PARQUET_CV_ANUAL = "cv_core_anual.parquet"

ORDEM_BUCKET = ['nano', 'micro', 'mid-tier', 'macro', 'mega']

## 2. Carga das Bases

In [34]:
base = pl.read_parquet(PARQUET_BASE_2017_2019)
core = pl.read_parquet(PARQUET_CORE_2017_2019)
cv_core = pl.read_parquet(PARQUET_CV_CORE)
cv_anual = pl.read_parquet(PARQUET_CV_ANUAL)

print(f"Base 2017-2019 (Principal): {base.shape}")
print(f"Core 2017-2019 (Robustez): {core.shape}")

Base 2017-2019 (Principal): (8897233, 32)
Core 2017-2019 (Robustez): (825268, 32)


## 3. Perfil Descritivo (Base Principal)

In [35]:
print("N Posts:", base.shape[0])
print("N Perfis Únicos:", base["username"].n_unique())
print("Período:", base["data_dt"].min(), "a", base["data_dt"].max())

N Posts: 8897233
N Perfis Únicos: 33149
Período: 2017-01-01 a 2019-05-15


In [36]:
base.group_by("post_type").agg(pl.len().alias("count")).with_columns((pl.col("count")/pl.col("count").sum()*100).alias("pct"))

post_type,count,pct
str,u32,f64
"""image""",7788045,87.533338
"""carousel""",1109188,12.466662


In [37]:
base.group_by("is_sponsored").agg(pl.len().alias("count"))

is_sponsored,count
bool,u32
false,8370573
true,526660


## 4. Engajamento Descritivo

In [38]:
base.select(["likes", "comments_count"]).describe()

statistic,likes,comments_count
str,f64,f64
"""count""",8.897233e6,8.897233e6
"""null_count""",0.0,0.0
"""mean""",4536.851643,7.237498
"""std""",39619.919914,327.183977
"""min""",0.0,0.0
"""25%""",151.0,0.0
"""50%""",496.0,0.0
"""75%""",1609.0,0.0
"""max""",8.444365e6,466119.0


In [39]:
base.select(["er_classico", "er_weighted"]).describe()

statistic,er_classico,er_weighted
str,f64,f64
"""count""",8.897233e6,8.897233e6
"""null_count""",0.0,0.0
"""mean""",4.327969,4.349675
"""std""",5.291196,5.33094
"""min""",0.0,0.0
"""25%""",1.569472,1.575025
"""50%""",3.020962,3.033254
"""75%""",5.491529,5.516313
"""max""",1408.70425,1408.70425


## 5. Patrocínio, Tipo de Post e Categoria

In [40]:
base.group_by("is_sponsored").agg(pl.col("er_classico").median().alias("er_mediano"))

is_sponsored,er_mediano
bool,f64
false,3.041145
true,2.730474


In [41]:
base.group_by(["inf_category", "post_type"]).agg(pl.col("er_classico").median().alias("er_mediano"))

inf_category,post_type,er_mediano
str,str,f64
"""family""","""image""",3.238273
"""food""","""carousel""",2.467918
"""travel""","""image""",3.716941
"""pet""","""carousel""",3.964401
"""fashion""","""image""",3.432647
…,…,…
"""fitness""","""image""",3.054983
"""food""","""image""",2.452279
"""beauty""","""image""",2.877346


## 6. Segmentação por Followers

In [42]:
base.group_by("bucket_followers").agg(pl.col("er_classico").median().alias("er_mediano"))

bucket_followers,er_mediano
str,f64
"""macro""",2.332092
"""mid-tier""",2.447922
"""nano""",3.808718
"""mega""",2.157625
"""micro""",2.789235


In [43]:
sweet_spot = (
    base.group_by(["inf_category", "bucket_followers"])
    .agg(pl.col("er_classico").median().alias("er_mediano"))
).to_pandas().pivot_table(index="inf_category", columns="bucket_followers", values="er_mediano")[ORDEM_BUCKET]
sweet_spot

bucket_followers,nano,micro,mid-tier,macro,mega
inf_category,,,,,
beauty,3.426791,2.677871,2.472974,3.034087,2.562905
family,3.746275,2.974897,2.832072,3.016734,2.414171
fashion,5.501121,3.130261,2.620683,2.800539,2.346211
fitness,4.232425,3.020036,2.189875,2.072940,2.555870
food,3.250466,2.132007,1.584516,0.974844,0.520989
interior,3.631600,2.173102,1.688036,0.933601,0.837626
other,2.227171,1.660356,2.003805,1.424658,1.653575
pet,4.507946,3.438325,3.586998,2.429313,2.238662
travel,4.794114,3.342060,2.733177,2.191942,2.334272


## 7. Padrões Temporais

In [44]:
base.with_columns(pl.col("data_dt").dt.year().alias("ano")).group_by("ano").agg(pl.col("er_classico").median().alias("er_mediano")).sort("ano")

ano,er_mediano
i32,f64
2017,2.941176
2018,3.134832
2019,2.775218


## 8. Correlações com er_classico

In [45]:
variaveis = ["n_hashtags", "caption_len", "n_usertags", "aspect_ratio", "n_imagens"]
df_corr = base.select(["er_classico"] + variaveis).drop_nulls().to_pandas()

resultados = []
for var in variaveis:
    rho_spearman, pval_s = spearmanr(df_corr["er_classico"], df_corr[var])
    rho_pearson, pval_p = pearsonr(df_corr["er_classico"], df_corr[var])
    resultados.append({
        "variavel": var, 
        "spearman_rho": round(rho_spearman, 4),
        "pearson_r": round(rho_pearson, 4)
    })

pd.DataFrame(resultados).sort_values("spearman_rho", key=abs, ascending=False)

,variavel,spearman_rho,pearson_r
2,n_usertags,0.1611,0.0948
3,aspect_ratio,-0.1237,-0.0526
0,n_hashtags,-0.0337,0.0135
1,caption_len,-0.0165,-0.0000
4,n_imagens,0.0152,0.0075


## 9. Estabilidade no Core 2017-2019
Esta seção utiliza exclusivamente a base `core_2017_2019` para análise de estabilidade e o CV calculado no pré-processamento.

In [46]:
cv_core.select(["er_media", "er_std", "er_mediana", "cv"]).describe()

statistic,er_media,er_std,er_mediana,cv
str,f64,f64,f64,f64
"""count""",32646.0,32646.0,32646.0,32646.0
"""null_count""",0.0,0.0,0.0,0.0
"""mean""",4.74545,2.340942,4.271269,0.505005
"""std""",3.974985,3.203497,3.600505,0.278049
"""min""",0.004926,0.002358,0.004302,0.020832
"""25%""",2.176713,0.922022,1.888922,0.330773
"""50%""",3.695104,1.616787,3.294679,0.44293
"""75%""",6.1254,2.759857,5.557828,0.601175
"""max""",91.207135,155.040067,62.667608,4.321113


In [47]:
perfis_estaveis = cv_core.sort("cv").head(10)
perfis_estaveis

username,n_posts_validos,er_media,er_std,er_mediana,cv
str,u32,f64,f64,f64,f64
"""molliejanegarza""",5,5.028249,0.104749,5.014124,0.020832
"""marta__lu""",11,1.234287,0.029774,1.231795,0.024123
"""thelagirl""",22,3.397562,0.094192,3.390797,0.027723
"""mr_youngblood_fitness""",25,4.622572,0.137349,4.622572,0.029713
"""gingerrlu""",19,4.074444,0.121273,4.025191,0.029764
"""womensfashionmode""",14,0.946024,0.029536,0.946365,0.031221
"""tazerleejones""",7,9.06114,0.392079,9.050577,0.04327
"""beautygala""",22,1.146375,0.05487,1.147863,0.047864
"""elainerau""",24,9.173139,0.444148,8.985,0.048418


In [48]:
# Distribuição do CV por categoria, ligando de volta com a informação do usuário na base core
cv_com_cat = cv_core.join(core.select(["username", "inf_category", "bucket_followers"]).unique(subset=["username"]), on="username", how="left")
cv_com_cat.group_by("inf_category").agg(pl.col("cv").median().alias("cv_mediano")).sort("cv_mediano")

inf_category,cv_mediano
str,f64
"""pet""",0.365354
"""travel""",0.398869
"""fashion""",0.411468
"""food""",0.44117
"""family""",0.449806
"""fitness""",0.456726
"""interior""",0.484893
"""beauty""",0.495626
"""other""",0.53747


## 10. Síntese Metodológica
- A inclusão dos posts sem comentários preservou quase a totalidade da base observacional natural de 2017-2019.
- O engajamento no período apresenta variações consistentes com patrocínio e escala da audiência (bucket).
- O uso da camada `core_2017_2019` para CV provê uma métrica de estabilidade robusta isolada de viés comportamental de engajamento nulo.